<div style="background: linear-gradient(120deg, #1ca9c9 0%, #40c4d6 60%, #7fe0e0 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Actividad 3: Extracción de archivos Parquet y JSON</h1>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Laboratorio de Procesamiento de Datos</h3>
    </div>
</div>


# 1. De múltiples CSV a un dataset Parquet analítico

Usando los archivos de la carpeta `Data/csvs_ejemplo/`, el estudiante deberá:

1. **Consolidar** todos los CSV en un solo `DataFrame`.
2. **Limpiar** los datos (nulos, duplicados y columnas índice).
3. Añadir una columna `origen` con el nombre del archivo de procedencia.
4. Guardar como **Parquet particionado por `Ciudad`**.
5. Comparar **tamaño en disco** y **tiempo de lectura** entre CSV y Parquet.
6. Responder:
   - ¿Cuál es el promedio de `Números` por ciudad?
   - ¿Qué ciudad tiene más registros?
   - ¿Cuánto espacio ahorra Parquet frente a CSV?

In [2]:
from pathlib import Path
import pandas as pd
import os
from time import time

base = Path("Data")
csv_dir = base / "csvs_ejemplo"

# 1 Consolidar todos los CSV y
dfs = []
for archivo in sorted(csv_dir.glob("*.csv")):
    df = pd.read_csv(archivo)
    df.columns = [str(c).strip() for c in df.columns]
    df = df.drop(columns=[c for c in df.columns if "Unnamed" in c], errors="ignore")
    df = df.dropna().drop_duplicates()
    df["origen"] = archivo.name
    dfs.append(df)

df_total = pd.concat(dfs, ignore_index=True)
df_total.head()

,Nombre,Números,Ciudad,origen
0,Ana,10,Guadalajara,archivo_1.csv
1,Beto,25,Monterrey,archivo_1.csv
2,Carla,17,Ciudad de México,archivo_1.csv
3,Diego,32,Puebla,archivo_1.csv
4,Elena,8,Querétaro,archivo_2.csv


In [3]:
# 2. Limpieza: quitar columnas índice, nulos y duplicados
df_total = df_total.rename(columns={"Números": "Numeros"})
df_total = df_total.dropna().drop_duplicates()
df_total = df_total.reset_index(drop=True)
df_total.head()

,Nombre,Numeros,Ciudad,origen
0,Ana,10,Guadalajara,archivo_1.csv
1,Beto,25,Monterrey,archivo_1.csv
2,Carla,17,Ciudad de México,archivo_1.csv
3,Diego,32,Puebla,archivo_1.csv
4,Elena,8,Querétaro,archivo_2.csv


In [4]:
# 3. Añadir la columna 'origen'
df_total[["Nombre", "Numeros", "Ciudad", "origen"]].head()

,Nombre,Numeros,Ciudad,origen
0,Ana,10,Guadalajara,archivo_1.csv
1,Beto,25,Monterrey,archivo_1.csv
2,Carla,17,Ciudad de México,archivo_1.csv
3,Diego,32,Puebla,archivo_1.csv
4,Elena,8,Querétaro,archivo_2.csv


In [ ]:
# 4. Dataset Parquet particionado por Ciudad
salida = base / "salida_parquet"
salida.mkdir(exist_ok=True)
df_total.to_parquet(salida / "dataset_total.parquet", index=False)
df_total.to_parquet(salida, index=False, partition_cols=["Ciudad"])
print("Parquet guardado en:", salida)

Parquet guardado en: Data\salida_parquet


In [6]:
# 5 comparar tamaño y tiempo de lectura CSV vs Parquet.
# Hint: Para medir el tiempo de lectura, puedes usar la función time.time() antes y después de la lectura del archivo, y calcular la diferencia. Por ejemplo:


#from time import time
#start_time = time.time()
# Código para leer el archivo
#end_time = time.time()
#print(f"Tiempo de lectura: {end_time - start_time} segundos")# 5. Comparar tamaño y tiempo de lectura CSV vs Parquet

csv_size = sum(p.stat().st_size for p in csv_dir.glob("*.csv"))
parquet_size = sum(p.stat().st_size for p in salida.rglob("*.parquet"))

start = time()
csv_df = pd.concat([pd.read_csv(p) for p in sorted(csv_dir.glob("*.csv"))], ignore_index=True)
csv_time = time() - start

start = time()
parquet_df = pd.read_parquet(salida / "dataset_total.parquet")
parquet_time = time() - start

print(f"Tamaño CSV total: {csv_size} bytes")
print(f"Tamaño Parquet total: {parquet_size} bytes")
print(f"Ahorrado: {((csv_size - parquet_size) / csv_size) * 100:.2f}%")
print(f"Tiempo lectura CSV: {csv_time:.4f} s")
print(f"Tiempo lectura Parquet: {parquet_time:.4f} s")



Tamaño CSV total: 575 bytes
Tamaño Parquet total: 43844 bytes
Ahorrado: -7525.04%
Tiempo lectura CSV: 0.0141 s
Tiempo lectura Parquet: 0.0133 s


In [7]:
# 6. Código y Respuestas de análisis

# Promedio de Numeros por ciudad
promedio = df_total.groupby("Ciudad")["Numeros"].mean()
print("Promedio por ciudad:")
print(promedio)

# Ciudad con más registros
ciudad_mas_registros = df_total["Ciudad"].value_counts()
print("\nCiudad con más registros:")
print(ciudad_mas_registros.head())

# Resultado final
print("\nDataset final:")
display(df_total.head())

Promedio por ciudad:
Ciudad
Aguascalientes      16.00
Cancún              23.00
Ciudad de México    29.00
Guadalajara         25.75
Hermosillo          27.00
León                13.00
Monterrey           31.00
Morelia              9.00
Mérida              29.00
Oaxaca              30.00
Puebla              25.50
Querétaro            8.00
Saltillo            34.00
Tijuana             21.00
Toluca              14.00
Veracruz            18.00
Name: Numeros, dtype: float64

Ciudad con más registros:
Ciudad
Guadalajara         4
Monterrey           2
Ciudad de México    2
Puebla              2
Querétaro           1
Name: count, dtype: int64

Dataset final:


,Nombre,Numeros,Ciudad,origen
0,Ana,10,Guadalajara,archivo_1.csv
1,Beto,25,Monterrey,archivo_1.csv
2,Carla,17,Ciudad de México,archivo_1.csv
3,Diego,32,Puebla,archivo_1.csv
4,Elena,8,Querétaro,archivo_2.csv



# 2. Extracción de archivos JSON

Utilizando el fichero `metallica.json` que se encuentra en la carpeta Data, que contiene información sobre los discos del grupo Metallica, crea distintas funciones en python, utilizando la librería `json`, que realicen las siguientes funciones:

1. **Listar información:** Listar el título del disco, y el contenido y fecha en el que se ha publicado en la wiki.
2. **Contar información:** Mostrar los títulos de discos y el número de canciones que tiene cada una.
3. **Buscar o filtrar información:** Recibe un límite inferior y otro superior, y muestra el título, y los nombres de las etiquetas ("tags") de los discos cuyos número de oyentes esté entre los dos límites.
4. **Buscar información relacionada:** Recibe el nombre una canción y muestra el título del disco donde está.

In [8]:
import json
from pathlib import Path

data_path = Path("Data/metallica.json")

with open(data_path, encoding="utf-8") as f:
    data = json.load(f)

print(type(data), len(data))

<class 'list'> 4


In [ ]:
#1. Función paara lsitar el titulo del disco, y el contenido y fecha en el que se ha publicado en la wiki# 1. Listar información: título del disco, contenido y fecha de la wiki
for album in data:
    info = album["album"]
    wiki = info.get("wiki", {})
    print("Álbum:", info["name"])
    print("Publicado:", wiki.get("published"))
    print("Resumen:", wiki.get("content", "")[:180])
    print("\n")

Álbum: Kill 'Em All
Publicado: 01 Apr 2009, 22:18
Resumen: Kill 'Em All is Metallica's first studio album. Some of the songs were written by a different line-up than the musicians that played the songs in the studio. It was recorded at Mus


Álbum: Ride the Lightning
Publicado: 26 Nov 2008, 18:21
Resumen: Ride The Lightning is the second album by thrash metal band Metallica. It was released on July 27, 1984 by Megaforce Records. It was re-released in the same year on November 19 by 


Álbum: Master of Puppets
Publicado: 25 Jul 2008, 02:55
Resumen: Master of Puppets is the third studio album by American heavy metal band Metallica. The album was recorded in 1985 and released by Elektra Records on March 3, 1986 in North America


Álbum: And Justice For All
Publicado: 12 Jun 2012, 04:07
Resumen: …And Justice for All is the fourth studio album by American heavy metal band Metallica. Elektra Records released the album on August 25, 1988. This is the first Metallica studio al




In [12]:
#2. Mostrar los títulos de discos y el número de canciones que tiene cada una.
for album in data:
    info = album["album"]
    tracks = info.get("tracks", {}).get("track", [])
    print(f"{info['name']} - {len(tracks)} canciones")

Kill 'Em All - 10 canciones
Ride the Lightning - 8 canciones
Master of Puppets - 8 canciones
And Justice For All - 9 canciones


In [13]:
#3. Recibe un límite inferior y otro superior, y muestra el título, y los nombres de las etiquetas ("tags") 
#   de los discos cuyos número de oyentes esté entre los dos límites.
def discos_por_oyentes(inferior, superior):
    resultados = []
    for album in data:
        info = album["album"]
        listeners = int(info.get("listeners", 0))
        if inferior <= listeners <= superior:
            tags = [t.get("name", "") for t in info.get("tags", {}).get("tag", [])]
            resultados.append((info["name"], tags))
    return resultados

print(discos_por_oyentes(500000, 700000))

[("Kill 'Em All", ['thrash metal', 'albums I own', 'metal', 'heavy metal', 'speed metal'])]


In [14]:
# 4. Recibe el nombre una canción y muestra el título del disco donde está.
def disco_por_cancion(nombre_cancion):
    for album in data:
        info = album["album"]
        tracks = info.get("tracks", {}).get("track", [])
        for track in tracks:
            if track.get("name", "").lower() == nombre_cancion.lower():
                return info["name"]
    return "No encontrada"

print(disco_por_cancion("Enter Sandman"))

No encontrada
